# Neural Networks — Statistical Learning

A leveled exercise. Most topics can be solved at **three levels** — but only the **foundational** ones offer all three; later, applied topics jump straight to the practical level. Pick the level that fits your comfort.

## How this notebook works

<table>
<tr><td style="font-size:1.4em;">🧮</td><td><b style="color:#E37222;">Level 1 · By hand</b></td><td>Do the math yourself on small <b>mock data</b> — plain arithmetic, no libraries.</td></tr>
<tr><td style="font-size:1.4em;">🐍</td><td><b style="color:#0065BD;">Level 2 · With NumPy</b></td><td>Implement the formula with basic <b>NumPy</b> on the real data.</td></tr>
<tr><td style="font-size:1.4em;">📚</td><td><b style="color:#5E6A00;">Level 3 · With libraries</b></td><td>Use <b>scikit-learn</b> — the practical way.</td></tr>
</table>

When you get stuck, expand a hint: <b>💡 Hint</b> points you to the right functions; <b>🔑 Stuck?</b> shows them on tiny mock data.

## Setup &mdash; packages and data

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

plt.style.use(os.path.join(os.getcwd(), "..", "..", "configs", "visualisations.mplstyle"))
df = pd.read_csv(os.path.join("..", "..", "data", "1900_2021_DISASTERS.csv"))
df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

types = ['Earthquake', 'Flood', 'Storm', 'Volcanic activity']
geo = df[df['Latitude'].between(-90, 90) & df['Longitude'].between(-180, 180)]
geo = geo[geo['Disaster Type'].isin(types)].copy()

X = geo[['Longitude', 'Latitude']].values.astype(float)
y = geo['Disaster Type'].astype(str).values
Xs = StandardScaler().fit_transform(X)                 # scaled inputs (nets train better on these)

from sklearn.preprocessing import LabelEncoder
y_code = LabelEncoder().fit_transform(y)               # integer class labels 0..3 for scikit-learn

# a train/test split for the library tasks
Xtr, Xte, ytr, yte = train_test_split(Xs, y_code, test_size=0.25, random_state=0, stratify=y_code)

# a binary sub-problem for the by-hand / NumPy parts: Earthquake vs Flood
bin_mask = np.isin(y, ['Earthquake', 'Flood'])
Xb = Xs[bin_mask]
yb = (y[bin_mask] == 'Flood').astype(int)              # 1 = Flood, 0 = Earthquake

print(geo.shape, '| classes:', list(np.unique(y)))
geo.head()

# 1 · Motivation — why go nonlinear

### A line is not enough

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Fit a linear model (<code>LogisticRegression</code>) and a neural network (<code>MLPClassifier</code> with two hidden layers) on the disaster data and compare their test accuracy. Which one captures the nonlinear class regions?</span></div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print(f'logistic regression: {acc_lin:.3f}   neural network: {acc_mlp:.3f}')

# 2 · Network architecture

### Activation functions

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">Evaluate by hand and fill in the values: sigmoid(0), tanh(0), ReLU(&minus;3), ReLU(2). Recall sigmoid(z) = 1 / (1 + e<sup>&minus;z</sup>).</span></div>

In [ ]:
# no libraries needed - reason it out
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print(sigmoid_at_0, tanh_at_0, relu_at_neg3, relu_at_2)

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement <code>sigmoid</code>, <code>tanh</code>, <code>relu</code> and <code>softmax</code> as NumPy functions. Make <code>softmax</code> numerically stable and normalise along the last axis.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Use [`np.exp`](https://numpy.org/doc/stable/reference/generated/numpy.exp.html), [`np.maximum`](https://numpy.org/doc/stable/reference/generated/numpy.maximum.html) for ReLU, and subtract `np.max(z, axis=-1, keepdims=True)` before exponentiating in softmax.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
z = np.array([2.0, 1.0, 0.1])
e = np.exp(z - z.max())
print(e / e.sum())   # sums to 1
```

</details>

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0, z)
def softmax(z):
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
z = np.array([-2.0, 0.0, 3.0])
print('sigmoid:', np.round(sigmoid(z), 3))
print('softmax sums to:', round(float(softmax(np.array([2.0, 1.0, 0.1])).sum()), 3))

### The forward pass

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">One neuron, two inputs. With weights w = [0.5, &minus;1.0], bias b = 0, and input x = [2.0, 1.0], compute z = w&middot;x + b by hand, then a = sigmoid(z).</span></div>

In [ ]:
# w = [0.5, -1.0], b = 0.0, x = [2.0, 1.0]
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print('z =', z_by_hand, ' a =', a_by_hand)

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Run one forward pass of a small network: 2 inputs &rarr; 3 hidden units (tanh) &rarr; 4 outputs (softmax). Use the random weights below and the first data sample; return the class probabilities.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

A layer is `activation(x @ W + b)`. Matrix product is `@`. Reuse your `tanh` and `softmax`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
h = tanh(x @ W1 + b1)     # (3,)
logits = h @ W2 + b2      # (4,)
probs = softmax(logits)
```

</details>

In [ ]:
rng = np.random.default_rng(0)
W1, b1 = rng.normal(size=(2, 3)), np.zeros(3)
W2, b2 = rng.normal(size=(3, 4)), np.zeros(4)
x = Xs[0]                                        # one (scaled) sample
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print('probs:', np.round(probs, 3), ' sum:', round(float(probs.sum()), 3))

### Output layer & softmax

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Given the four class scores (logits) below, turn them into probabilities with your <code>softmax</code> and read off the predicted class from <code>types</code>.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

`np.argmax` gives the index of the largest probability; index into `types`.

</details>

In [ ]:
logits = np.array([3.1, 0.7, -0.4, 1.2])         # scores for [EQ, Flood, Storm, Volcanic]
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print('probabilities:', np.round(p, 3), ' sum:', round(float(p.sum()), 3))
print('predicted class:', pred_class)

# 3 · Loss and backpropagation

### Cross-entropy loss

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">A softmax output is p = [0.70, 0.10, 0.05, 0.15] for classes [EQ, Flood, Storm, Volcanic]. The true class is <b>Flood</b>. Compute the cross-entropy for this sample, &minus;log(p<sub>true</sub>), by hand.</span></div>

In [ ]:
import math
# true class is Flood -> its probability is 0.10
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print('cross-entropy:', round(ce_by_hand, 3))

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Implement the mean cross-entropy over a batch: given softmax rows <code>P</code> (n&times;K) and one-hot targets <code>Y</code> (n&times;K), return &minus;mean(&Sigma; Y&middot;log P).</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

One-hot `Y` selects the true-class log-probability. Add a tiny `eps` inside the log for safety.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
eps = 1e-12
row_loss = -np.sum(Y * np.log(P + eps), axis=1)
return row_loss.mean()
```

</details>

In [ ]:
def cross_entropy(P, Y):
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
P = np.array([[0.7, 0.1, 0.05, 0.15], [0.1, 0.2, 0.6, 0.1]])
Y = np.array([[0, 1, 0, 0],           [0, 0, 1, 0]])
print('mean cross-entropy:', round(cross_entropy(P, Y), 3))

### The output gradient

<div style="background-color:#FCEEE3; border-left:6px solid #E37222; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#E37222;">🧮&nbsp; Level 1 · By hand</b><br><span style="color:#000;">Single sigmoid output. The network predicts A = 0.8 (its P(Flood)) and the true label is y = 1. The gradient of the binary cross-entropy w.r.t. the logit z is &part;R/&part;z = A &minus; y. Fill it in.</span></div>

In [ ]:
# A = 0.8, y = 1
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
print('dR/dz =', dR_dz)

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">For binary cross-entropy R(z) = &minus;[y·log s + (1&minus;y)·log(1&minus;s)] with s = sigmoid(z), implement the analytic gradient <code>grad_bce(z, y)</code> and check it against a finite difference.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

The algebra collapses to `sigmoid(z) - y`. Compare with a central difference of `bce`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
num = (bce(z+1e-5, y) - bce(z-1e-5, y)) / (2e-5)
```

</details>

In [ ]:
def bce(z, y):
    s = sigmoid(z); eps = 1e-12
    return -(y * np.log(s + eps) + (1 - y) * np.log(1 - s + eps))

def grad_bce(z, y):
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
z0, y0 = 0.4, 1.0
numeric = (bce(z0 + 1e-5, y0) - bce(z0 - 1e-5, y0)) / (2e-5)
print('analytic:', round(float(grad_bce(z0, y0)), 4), ' numeric:', round(float(numeric), 4))

# 4 · Optimization

### Gradient descent

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Fit a binary logistic model P(Flood) = sigmoid(X·w + b) on the subset (<code>Xb</code>, <code>yb</code>) with full-batch gradient descent. Fill in the gradient step and return the loss curve.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

For log-loss the gradient is `X.T @ (s - y) / n` for the weights and `mean(s - y)` for the bias.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
err = s - y
w -= rho * (X.T @ err / n)
b -= rho * err.mean()
```

</details>

In [ ]:
def fit_gd(X, y, rho=0.3, steps=300):
    n, d = X.shape
    w, b, losses = np.zeros(d), 0.0, []
    for _ in range(steps):
        s = sigmoid(X @ w + b)
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
        losses.append(bce(X @ w + b, y).mean())
    return w, b, losses

w, b, losses = fit_gd(Xb, yb)
plt.plot(losses); plt.xlabel('step'); plt.ylabel('BCE loss'); plt.title('gradient descent'); plt.show()
print('final loss:', round(losses[-1], 3))

### Learning rate

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Run <code>fit_gd</code> with several learning rates and plot the loss curves on one axis. Which &rho; is too small, which is well chosen, which is too large?</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Loop over a list of rates and call `fit_gd(Xb, yb, rho=...)` for each.

</details>

In [ ]:
plt.figure()
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
plt.legend(); plt.xlabel('step'); plt.ylabel('BCE loss'); plt.ylim(0, 1); plt.title('learning rate'); plt.show()

### Stochastic & mini-batch descent

<div style="background-color:#E8F1FA; border-left:6px solid #0065BD; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#0065BD;">🐍&nbsp; Level 2 · With NumPy</b><br><span style="color:#000;">Turn gradient descent into <b>mini-batch</b> descent: each epoch, shuffle the data and take one step per batch. Compare the loss curves for batch sizes 1 (stochastic), 32 and the full set.</span></div>

<details>
<summary>💡 <b>Hint</b> &mdash; which functions to use</summary>

Shuffle indices with `np.random.default_rng().permutation(n)`, then slice out batches of size `batch`.

</details>

<details>
<summary>🔑 <b>Stuck?</b> Worked example</summary>

```python
for s0 in range(0, n, batch):
    bi = idx[s0:s0+batch]
    s = sigmoid(X[bi] @ w + b)
    err = s - y[bi]
    w -= rho * (X[bi].T @ err / len(bi))
    b -= rho * err.mean()
```

</details>

In [ ]:
def fit_minibatch(X, y, rho=0.3, epochs=40, batch=32, seed=0):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w, b, losses = np.zeros(d), 0.0, []
    for _ in range(epochs):
        idx = rng.permutation(n)
        for s0 in range(0, n, batch):
            bi = idx[s0:s0 + batch]
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
        losses.append(bce(X @ w + b, y).mean())
    return losses

plt.figure()
for B in [1, 32, len(Xb)]:
    plt.plot(fit_minibatch(Xb, yb, batch=B), label=f'batch={B}')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('BCE loss'); plt.title('batch size'); plt.show()

### Momentum & Adam

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Train two networks with scikit-learn on the 4-class problem: one with <code>solver='sgd'</code> and <code>momentum=0.9</code>, one with <code>solver='adam'</code>. Plot their <code>loss_curve_</code> and compare test accuracy.</span></div>

In [ ]:
from sklearn.neural_network import MLPClassifier
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
plt.plot(sgd.loss_curve_,  label='SGD + momentum')
plt.plot(adam.loss_curve_, label='Adam')
plt.legend(); plt.xlabel('iteration'); plt.ylabel('training loss'); plt.title('optimizers'); plt.show()
print(f'test accuracy   sgd {sgd.score(Xte, yte):.3f}   adam {adam.score(Xte, yte):.3f}')

# 5 · Regularization and tuning

### Overfitting: training vs validation

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Train a fairly large network epoch by epoch (<code>warm_start=True</code>, <code>max_iter=1</code>) and record training and validation accuracy after each epoch. Plot both — where do they diverge?</span></div>

In [ ]:
from sklearn.neural_network import MLPClassifier
Xt, Xv, yt, yv = train_test_split(Xtr, ytr, test_size=0.3, random_state=0, stratify=ytr)
clf = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=1, warm_start=True, random_state=0)
tr_acc, va_acc = [], []
for _ in range(120):
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---
plt.plot(tr_acc, label='train'); plt.plot(va_acc, label='validation')
plt.legend(); plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.title('overfitting'); plt.show()

### Weight decay & early stopping

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Sweep the L2 penalty <code>alpha</code> (weight decay) and turn on <code>early_stopping</code>. Report the test accuracy for each &mdash; too little regularization overfits, too much underfits.</span></div>

In [ ]:
for a in [1e-5, 1e-3, 1e-1, 1.0]:
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---

### Architecture & hyperparameters

<div style="background-color:#F1F3DD; border-left:6px solid #5E6A00; padding:8px 14px; border-radius:6px; margin:6px 0;"><b style="font-size:1.05em; color:#5E6A00;">📚&nbsp; Level 3 · With libraries</b><br><span style="color:#000;">Compare a few architectures (width and depth) at a fixed weight decay and report test accuracy. Bigger is not always better.</span></div>

In [ ]:
for hidden in [(8,), (32, 16), (64, 64, 64)]:
# ---- YOUR CODE GOES HERE (START) ---

# --- YOUR CODE GOES HERE (END) ---